# Väljakutse: Andmeteaduse teksti analüüs

Selles näites teeme lihtsa harjutuse, mis hõlmab kõiki traditsioonilise andmeteaduse protsessi samme. Sa ei pea kirjutama mingit koodi, saad lihtsalt allolevatel lahtritel klikata, et neid käivitada ja tulemust vaadata. Väljakutsena soovitatakse proovida seda koodi erinevate andmetega.

## Eesmärk

Selles õppetükis oleme arutanud erinevaid andmeteadusega seotud mõisteid. Proovime avastada rohkem seotud mõisteid, tehes **tekstikaevandust**. Alustame tekstiga andmeteadusest, väljavõtame sellest märksõnad ja proovitame tulemuse visualiseerida.

Tekstina kasutan Wikipedia andmeteaduse lehte:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Samm 1: Andmete hankimine

Iga andmeteaduse protsessi esimene samm on andmete hankimine. Selleks kasutame `requests` teeki:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Samm 2: Andmete töötlemine

Järgmine samm on andmete teisendamine protsessimiseks sobivasse vormingusse. Meie puhul oleme lehelt alla laadinud HTML algkoodi ja peame selle teisendama lihttekstiks.

Seda saab teha mitmel moel. Me kasutame [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), populaarset Python'i teeki HTML-i töötlemiseks. BeautifulSoup võimaldab meil sihtida konkreetseid HTML elemente, et keskenduda Wikipedia põhisisule ja vähendada mõningaid navigeerimismenüüsid, külgribasid, jaluseid ja muud ebaolulist sisu (kuigi osa malli teksti võib siiski alles jääda).


Esiteks peame HTML-i parsimiseks paigaldama BeautifulSoupi teegi:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Samm 3: Tõlgenduste saamine

Kõige olulisem samm on andmete muutmine selliseks kujuks, millest saame tõlgendusi teha. Meie puhul tahame tekstist välja tuua märksõnad ja vaadata, millised märksõnad on tähendusrikkamad.

Kasutame märksõnade väljavõtmiseks Python'i teeki nimega [RAKE](https://github.com/aneesha/RAKE). Esiteks paigaldame selle teegi juhuks, kui seda veel ei ole: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Peamine funktsionaalsus on saadaval objektist `Rake`, mida saame kohandada mitmete parameetrite abil. Meie puhul seame märksõna miinimumpikkuseks 5 tähemärki, märksõna miinimumsageduseks dokumendis 3 ning märksõnas olevate sõnade maksimumarvuks 2. Mängi julgelt teiste väärtustega ja vaata tulemust.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Saime termini nimekirja koos seotud tähtsuse astmega. Nagu näha, on nimekirjas esikohal kõige asjakohasemad valdkonnad, nagu masinõpe ja suurandmed.

## Samm 4: Tulemus visualiseerimine

Inimesed suudavad andmeid kõige paremini tõlgendada visuaalses vormis. Seetõttu on sageli mõistlik andmeid visualiseerida, et teha mõningaid järeldusi. Saame kasutada Pythoni `matplotlib` teeki, et joonistada lihtne võtmesõnade ja nende asjakohasuse jaotus:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Veelgi parem viis sõnasageduste visualiseerimiseks on **Sõnapilve** kasutamine. Me peame installima teise teegi, et joonistada sõnapilv meie märksõnalistist.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` objekt vastutab kas originaalteksti või eelnevalt arvutatud sõnade sageduste nimekirja vastuvõtmise eest ning tagastab pildi, mida saab seejärel kuvada kasutades `matplotlib`-i:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Me võime samuti anda `WordCloud`-ile algse teksti - vaatame, kas suudame saada sarnase tulemuse:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Nüüd näete, et sõnapilv näeb välja muljetavaldavam, kuid sisaldab ka palju müra (nt mitteasuvaid sõnu nagu `Retrieved on`). Samuti saame vähem märksõnu, mis koosnevad kahest sõnast, näiteks *andmeteadlane* või *rakendusinformaatika*. See on sellepärast, et RAKE algoritm töötab tekstist head märksõnad välja valides palju paremini. See näide illustreerib andmete eeltöötluse ja puhastamise olulisust, sest selge pilt lõpus võimaldab meil teha paremaid otsuseid.

Selles harjutuses oleme läbinud lihtsa protsessi, kuidas tuua Wikipedia tekstist välja tähendust märksõnade ja sõnapilve vormis. See näide on üsna lihtne, kuid demonstreerib hästi kõiki tüüpilisi samme, mida andmeteadlane teeb andmetega töötades, alates andmete hankimisest kuni visualiseerimiseni.

Meie kursusel käsitleme kõiki neid samme üksikasjalikult.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
